In [ ]:
# Step 3a: Persistent water mask for all wetlands

In [ ]:
# ===============================================================
# Step-3 — Persistent water mask (ga_ls_wo_3) + Enc/Ret ablation
# Windows: fixed cadence (e.g., 5-yr) + custom hydrological epochs
# Period: 1988–2023
# Output: per-wetland table with BEFORE vs AFTER (WOfS) transitions
# ===============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
import datacube
from datacube.utils.geometry import Geometry, CRS
from datacube.utils import masking
from shapely.geometry import mapping
from datetime import datetime
from rasterio.enums import Resampling
from rasterio import features

# -------------------
# CONFIG
# -------------------
START_YEAR   = 1988
END_YEAR     = 2023

# Main cadence; re-run with 10 if you want decadal
INTERVAL_YRS = 5

# Hydrological epochs (edit as needed)
CUSTOM_EPOCHS = [
    {"label": "1988-1993_WetBaseline",        "start": 1988, "end": 1993},
    {"label": "1994-2000_EarlyDry",           "start": 1994, "end": 2000},
    {"label": "2001-2009_MillenniumDrought",  "start": 2001, "end": 2009},
    {"label": "2010-2012_PostDroughtFloods",  "start": 2010, "end": 2012},
    {"label": "2013-2017_ModerateDry",        "start": 2013, "end": 2017},
    {"label": "2018-2019_SevereDrought",      "start": 2018, "end": 2019},
    {"label": "2020-2022_LaNinaFloods",       "start": 2020, "end": 2022},
]

PIXEL_RES   = 30
PX_TO_HA    = (PIXEL_RES**2) / 10000.0
TARGET_CRS  = "EPSG:3577"
BUFFER_M    = 500

# WOfS persistence settings
WET_FREQ_THRESH = 80.0  # % of clear observations wet
MIN_CLEAR_OBS   = 20    # minimum clear observations to trust wetness frequency

BASE_DIR   = "/home/jovyan/DEA products paper/Data"
OUT_TABLES = "/home/jovyan/DEA products paper/Results/step 3_v3/tables"
os.makedirs(OUT_TABLES, exist_ok=True)

WETLAND_SHP_PATHS = [
    os.path.join(BASE_DIR, "wetland boundaries/P1ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P2ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P3ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P4ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P5ANAEWetlands.shp"),
]

LC_PRODUCT = "ga_ls_landcover_class_cyear_3"

# -------------------
# CLASS GROUPS (Natural-only + ambiguous NAT kept)
# -------------------
WOODY_CODES = {20,27,28,29,30,31, 56,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77}
HERB_CODES  = {21,32,33,34,35,36, 57,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92}
BARE_CODES  = {94,95,96,97}
WATER_CODES = {98,99,100,101,102,103,104}
AMBIG_NAT   = {19,22,23,24,25,26, 55,58,59,60,61,62}

# Exclusions from validity
ARTIFICIAL_CODES = {93}
NODATA_CODES     = {255}
CULTIVATED       = {1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18}
MASK_OUT_CODES   = ARTIFICIAL_CODES | NODATA_CODES | CULTIVATED

# -------------------
# HELPERS
# -------------------
def make_periods(start, end, interval_years=None, custom_epochs=None):
    periods = []
    if interval_years:
        years = list(range(start, end, interval_years))
        for y0 in years:
            y1 = y0 + interval_years
            if y1 <= end:
                periods.append((f"{y0}-{y1}", y0, y1))
    if custom_epochs:
        for ep in custom_epochs:
            label = ep["label"]
            y0, y1 = int(ep["start"]), int(ep["end"])
            if y0 < y1 and (start <= y0 < y1 <= end):
                periods.append((label, y0, y1))
    return periods

def dc_query_from_geom(geom, crs_str=TARGET_CRS, res=PIXEL_RES, buffer_m=BUFFER_M):
    geom_buf = geom.buffer(buffer_m)
    return {
        "geopolygon": Geometry(mapping(geom_buf), CRS(crs_str)),
        "output_crs": crs_str,
        "resolution": (-res, res),
    }

def mask_codes(da: xr.DataArray, keep: set) -> xr.DataArray:
    codes = np.array(list(keep), dtype="int32")
    return xr.apply_ufunc(np.isin, da.astype("int32"), codes,
                          dask="parallelized", output_dtypes=[bool])

def count_px(mask: xr.DataArray) -> int:
    return int(mask.sum().values)

# -------------------
# MAIN
# -------------------
periods = make_periods(START_YEAR, END_YEAR, INTERVAL_YRS, CUSTOM_EPOCHS)
print("Total periods (fixed + epochs):", len(periods))

dc = datacube.Datacube(app="WPE_LC_L4_Step3_WOfS")

# Load wetlands (merged to subregion)
wetland_list = []
for sub_idx, path in enumerate(WETLAND_SHP_PATHS, start=1):
    gdf = gpd.read_file(path).to_crs(TARGET_CRS)
    if "OBJECTID" not in gdf.columns:
        raise ValueError(f"{path} has no OBJECTID field!")
    gdf["WetlandID"] = gdf["OBJECTID"].astype(int)
    gdf["SubRegion"] = f"SR{sub_idx}"
    wetland_list.append(gdf)
wetlands = gpd.GeoDataFrame(pd.concat(wetland_list, ignore_index=True), crs=TARGET_CRS)

rows = []

for subr in wetlands["SubRegion"].unique():
    sub_wetlands = wetlands[wetlands["SubRegion"] == subr].reset_index(drop=True)
    print(f"▶ Processing {subr} ... ({len(sub_wetlands)} wetlands)")

    # Batch on subregion extent
    geom = sub_wetlands.unary_union
    q = dc_query_from_geom(geom)

    # --- Load Land Cover stack ---
    ds_lc = dc.load(
        product=LC_PRODUCT,
        measurements=["full_classification"],
        time=(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"),
        **q
    )
    if ds_lc.sizes.get("time", 0) == 0:
        continue
    lc = ds_lc["full_classification"]

    # --- Load WOfS time series and derive persistent-water mask ---
    ds_wo = dc.load(
        product="ga_ls_wo_3",
        measurements=["water"],
        time=(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"),
        group_by="solar_day",
        **q
    )
    if ds_wo.sizes.get("time", 0) > 0:
        water_da = ds_wo["water"]
        wet   = masking.make_mask(water_da, wet=True)
        dry   = masking.make_mask(water_da, dry=True)
        clear = wet | dry
        wet_count   = wet.astype("int16").sum("time")
        clear_count = clear.astype("int16").sum("time")
        with np.errstate(divide="ignore", invalid="ignore"):
            wet_freq_pct = (wet_count.astype("float32") /
                            xr.where(clear_count > 0, clear_count, np.nan)) * 100.0
    else:
        wet_freq_pct, clear_count = None, None

    # --- Rasterize WetlandID mask on the LC grid ---
    transform = lc.rio.transform()
    shape = lc.rio.shape
    wetid_mask = features.rasterize(
        [(geom, wid) for wid, geom in zip(sub_wetlands["WetlandID"], sub_wetlands.geometry)],
        out_shape=shape, transform=transform, fill=0, dtype="int32"
    )
    wetid_mask = xr.DataArray(wetid_mask, dims=("y","x"), coords={"y": lc.y, "x": lc.x})

    # --- Loop all periods ---
    for label, y0, y1 in periods:
        base = lc.sel(time=lc.time.dt.year == y0).squeeze(drop=True)
        tgt  = lc.sel(time=lc.time.dt.year == y1).squeeze(drop=True)
        if base.size == 0 or tgt.size == 0:
            continue

        # Validity: natural-only (keep ambiguous NAT), exclude cultivated/urban/nodata
        valid_base = ~mask_codes(base, MASK_OUT_CODES)
        valid_tgt  = ~mask_codes(tgt,  MASK_OUT_CODES)
        valid = valid_base & valid_tgt

        # Group masks (baseline/target)
        base_woody  = mask_codes(base, WOODY_CODES) & valid
        tgt_woody   = mask_codes(tgt,  WOODY_CODES) & valid

        base_herb   = mask_codes(base, HERB_CODES)  & valid
        base_bare   = mask_codes(base, BARE_CODES)  & valid
        base_water  = mask_codes(base, WATER_CODES) & valid
        base_ambig  = mask_codes(base, AMBIG_NAT)   & valid

        # --- Transitions BEFORE water mask (restricted to wetland pixels later per-wetland) ---
        enc_h = base_herb  & tgt_woody
        enc_b = base_bare  & tgt_woody
        enc_w = base_water & tgt_woody
        enc_a = base_ambig & tgt_woody
        enc_any = enc_h | enc_b | enc_w | enc_a

        ret_h = mask_codes(base, WOODY_CODES) & mask_codes(tgt, HERB_CODES)  & valid
        ret_b = mask_codes(base, WOODY_CODES) & mask_codes(tgt, BARE_CODES)  & valid
        ret_w = mask_codes(base, WOODY_CODES) & mask_codes(tgt, WATER_CODES) & valid
        ret_a = mask_codes(base, WOODY_CODES) & mask_codes(tgt, AMBIG_NAT)   & valid  # supplementary
        ret_any_clear = ret_h | ret_b | ret_w
        ret_any_ext   = ret_any_clear | ret_a

        # --- Build persistent water ablation mask on the period grid ---
        if wet_freq_pct is not None and clear_count is not None:
            wf = wet_freq_pct.rio.reproject_match(base, resampling=Resampling.nearest)
            cc = clear_count.rio.reproject_match(base, resampling=Resampling.nearest)
            water_mask = (wf > WET_FREQ_THRESH) & (cc >= MIN_CLEAR_OBS)
        else:
            water_mask = xr.zeros_like(base, dtype=bool)

        keep = (~water_mask)  # wetland ID selection is applied per-wetland below

        # After-mask transitions
        enc_h_a = enc_h & keep
        enc_b_a = enc_b & keep
        enc_w_a = enc_w & keep
        enc_a_a = enc_a & keep
        enc_any_a = enc_h_a | enc_b_a | enc_w_a | enc_a_a

        ret_h_a = ret_h & keep
        ret_b_a = ret_b & keep
        ret_w_a = ret_w & keep
        ret_a_a = ret_a & keep
        ret_any_clear_a = ret_h_a | ret_b_a | ret_w_a
        ret_any_ext_a   = ret_any_clear_a | ret_a_a

        # --- Per-wetland aggregation ---
        for wid in sub_wetlands["WetlandID"]:
            mask_w = (wetid_mask == wid)

            # Persistent water within this wetland (for info)
            persist_px = count_px(water_mask.where(mask_w))
            persist_ha = persist_px * PX_TO_HA

            # BEFORE counts
            eH_bef = count_px(enc_h.where(mask_w))
            eB_bef = count_px(enc_b.where(mask_w))
            eW_bef = count_px(enc_w.where(mask_w))
            eA_bef = count_px(enc_a.where(mask_w))
            eAny_bef = eH_bef + eB_bef + eW_bef + eA_bef

            rH_bef = count_px(ret_h.where(mask_w))
            rB_bef = count_px(ret_b.where(mask_w))
            rW_bef = count_px(ret_w.where(mask_w))
            rA_bef = count_px(ret_a.where(mask_w))
            rAnyClear_bef = rH_bef + rB_bef + rW_bef
            rAnyExt_bef   = rAnyClear_bef + rA_bef

            # AFTER counts
            eH_aft = count_px(enc_h_a.where(mask_w))
            eB_aft = count_px(enc_b_a.where(mask_w))
            eW_aft = count_px(enc_w_a.where(mask_w))
            eA_aft = count_px(enc_a_a.where(mask_w))
            eAny_aft = eH_aft + eB_aft + eW_aft + eA_aft

            rH_aft = count_px(ret_h_a.where(mask_w))
            rB_aft = count_px(ret_b_a.where(mask_w))
            rW_aft = count_px(ret_w_a.where(mask_w))
            rA_aft = count_px(ret_a_a.where(mask_w))
            rAnyClear_aft = rH_aft + rB_aft + rW_aft
            rAnyExt_aft   = rAnyClear_aft + rA_aft

            # Removed by WOfS (totals + split)
            enc_removed_px = eAny_bef - eAny_aft
            ret_removed_px = rAnyClear_bef - rAnyClear_aft
            enc_removed_ha = enc_removed_px * PX_TO_HA
            ret_removed_ha = ret_removed_px * PX_TO_HA

            # Record
            rows.append({
                "WetlandID": int(wid), "SubRegion": subr,
                "Period": label, "BaselineYear": y0, "TargetYear": y1,

                # Persistent water within wetland
                "PersistWater_px": persist_px,
                "PersistWater_ha": persist_ha,

                # --- Encroachment BEFORE (px & ha)
                "Enc_Herb_to_Woody_px_before": eH_bef,
                "Enc_Bare_to_Woody_px_before": eB_bef,
                "Enc_Water_to_Woody_px_before": eW_bef,
                "Enc_Ambig_to_Woody_px_before": eA_bef,
                "Enc_Any_to_Woody_px_before": eAny_bef,
                "Enc_Any_to_Woody_ha_before": eAny_bef * PX_TO_HA,

                # --- Encroachment AFTER (px & ha)
                "Enc_Herb_to_Woody_px_after": eH_aft,
                "Enc_Bare_to_Woody_px_after": eB_aft,
                "Enc_Water_to_Woody_px_after": eW_aft,
                "Enc_Ambig_to_Woody_px_after": eA_aft,
                "Enc_Any_to_Woody_px_after": eAny_aft,
                "Enc_Any_to_Woody_ha_after": eAny_aft * PX_TO_HA,

                # --- Retreat (CLEAR) BEFORE
                "Ret_Woody_to_Herb_px_before": rH_bef,
                "Ret_Woody_to_Bare_px_before": rB_bef,
                "Ret_Woody_to_Water_px_before": rW_bef,
                "Ret_Any_to_NonWoody_px_before": rAnyClear_bef,
                "Ret_Any_to_NonWoody_ha_before": rAnyClear_bef * PX_TO_HA,

                # --- Retreat (CLEAR) AFTER
                "Ret_Woody_to_Herb_px_after": rH_aft,
                "Ret_Woody_to_Bare_px_after": rB_aft,
                "Ret_Woody_to_Water_px_after": rW_aft,
                "Ret_Any_to_NonWoody_px_after": rAnyClear_aft,
                "Ret_Any_to_NonWoody_ha_after": rAnyClear_aft * PX_TO_HA,

                # --- Retreat (AMBIG supplementary) BEFORE/AFTER
                "Ret_Woody_to_Ambig_px_before": rA_bef,
                "Ret_Woody_to_Ambig_px_after": rA_aft,

                # --- Retreat (EXTENDED = clear + ambig)
                "Ret_Any_to_NonWoody_Extended_px_before": rAnyExt_bef,
                "Ret_Any_to_NonWoody_Extended_px_after": rAnyExt_aft,
                "Ret_Any_to_NonWoody_Extended_ha_before": rAnyExt_bef * PX_TO_HA,
                "Ret_Any_to_NonWoody_Extended_ha_after": rAnyExt_aft * PX_TO_HA,

                # --- Totals removed by water (helpful QC)
                "Enc_Removed_px": enc_removed_px,
                "Enc_Removed_ha": enc_removed_ha,
                "Ret_Removed_px": ret_removed_px,
                "Ret_Removed_ha": ret_removed_ha,
            })

# -------------------
# EXPORT
# -------------------
summary = pd.DataFrame(rows).sort_values(
    ["SubRegion","WetlandID","BaselineYear","TargetYear"]
).reset_index(drop=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(
    OUT_TABLES, f"wofs_mask_{START_YEAR}_{END_YEAR}_{INTERVAL_YRS}yr_plusEpochs_{ts}.csv"
)
summary.to_csv(csv_path, index=False)

print("✅ Exported water-mask summary to:", csv_path)
summary

